# Library & Config

In [1]:
import polars as pl
import numpy as np

# polars config
pl.Config.set_tbl_cols(-1)

polars.config.Config

# Load Lazy Dataset

In [2]:
TRAIN_RAW = "../data/raw/train.csv"
TEST_RAW = "../data/raw/test.csv"

missing_val = ["N/a", "n/a", "No", r"N\a", "N\\a", "na", "NA"]

# 1. SCAN
train_scan = pl.scan_csv(
    TRAIN_RAW,
    try_parse_dates=True,
    null_values=missing_val,
)

test_scan = pl.scan_csv(
    TEST_RAW,
    try_parse_dates=True,
    null_values=missing_val,
)

# 2. RENAME
train_trans = train_scan.rename(
    {
        column: column.lower()
        for column in train_scan.collect_schema().names()
    }
)

test_trans = test_scan.rename(
    {
        column: column.lower()
        for column in test_scan.collect_schema().names()
    }
)

# 3. CHANGE INTO NORMAL DATAFRAME
train = train_trans.collect()
test = test_trans.collect()

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (891, 12)
Test shape: (418, 11)


# Train Dataset

In [3]:
train.describe()

statistic,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
str,f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",891.0,891.0,891.0,"""891""","""891""",714.0,891.0,891.0,"""891""",891.0,"""204""","""889"""
"""null_count""",0.0,0.0,0.0,"""0""","""0""",177.0,0.0,0.0,"""0""",0.0,"""687""","""2"""
"""mean""",446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null
"""std""",257.353842,0.486592,0.836071,null,null,14.526497,1.102743,0.806057,null,49.693429,null,null
"""min""",1.0,0.0,1.0,"""Abbing, Mr. Anthony""","""female""",0.42,0.0,0.0,"""110152""",0.0,"""A10""","""C"""
"""25%""",224.0,0.0,2.0,null,null,20.0,0.0,0.0,null,7.925,null,null
"""50%""",446.0,0.0,3.0,null,null,28.0,0.0,0.0,null,14.4542,null,null
"""75%""",669.0,1.0,3.0,null,null,38.0,1.0,0.0,null,31.0,null,null
"""max""",891.0,1.0,3.0,"""van Melkebeke, Mr. Philemon""","""male""",80.0,8.0,6.0,"""WE/P 5735""",512.3292,"""T""","""S"""


## Dictionary

In [ ]:
# ambil dari data column di atas artinya tidak buat manual

column_metadata = {
    "PassengerId": {
        "description": "Unique passenger identifier",
        "role": "identifier",
    },
    "Survived": {
        "description": "Survival target",
        "role": "target",
        "key_means": "0 = No, 1 = Yes"
    },
    "Pclass": {
        "description": "Passenger ticket class",
        "role": "feature",
        "key_means": "1 = 1st, 2 = 2nd, 3 = 3rd"
    },
    "Name": {
        "description": "Passenger name",
        "role": "feature",
    },
    "Sex": {
        "description": "Passenger sex",
        "role": "feature",
    },
    "Age": {
        "description": "Passenger age",
        "role": "feature",
    },
    "SibSp": {
        "description": "Number of siblings/spouses aboard",
        "role": "feature",
    },
    "Parch": {
        "description": "Number of parents/children aboard",
        "role": "feature",
    },
    "Ticket": {
        "description": "Ticket number",
        "role": "feature",
    },
    "Fare": {
        "description": "Passenger fare",
        "role": "feature",
    },
    "Cabin": {
        "description": "Cabin number",
        "role": "feature",
    },
    "Embarked": {
        "description": "Port of embarkation",
        "role": "feature",
        "key_means": "C = Cherbourg, Q = Queenstown, S = Southampton"
    },
}

In [5]:
data_dictionary = pl.DataFrame(
    [
        {
            "column": column,
            **metadata,
        }
        for column, metadata in column_metadata.items()
    ]
)

with pl.Config(tbl_rows=-1):
    print(data_dictionary)

shape: (12, 4)
┌─────────────┬─────────────────────────────────┬────────────┬─────────────────────────────────┐
│ column      ┆ description                     ┆ role       ┆ key_means                       │
│ ---         ┆ ---                             ┆ ---        ┆ ---                             │
│ str         ┆ str                             ┆ str        ┆ str                             │
╞═════════════╪═════════════════════════════════╪════════════╪═════════════════════════════════╡
│ PassengerId ┆ Unique passenger identifier     ┆ identifier ┆ null                            │
│ Survived    ┆ Survival target                 ┆ target     ┆ 0 = No, 1 = Yes                 │
│ Pclass      ┆ Passenger ticket class          ┆ feature    ┆ 1 = 1st, 2 = 2nd, 3 = 3rd       │
│ Name        ┆ Passenger name                  ┆ feature    ┆ null                            │
│ Sex         ┆ Passenger sex                   ┆ feature    ┆ null                            │
│ Age         ┆

## Data validation

In [6]:
expected_columns = [
    "PassengerId",
    "Survived",
    "Pclass",
    "Name",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Ticket",
    "Fare",
    "Cabin",
    "Embarked",
]

assert train.columns == expected_columns

AssertionError: 

### Validasi tipe data

In [ ]:
assert train["PassengerId"].dtype == pl.Int64
assert train["Survived"].dtype == pl.Int64
assert train["Pclass"].dtype == pl.Int64

### Validasi nilai

In [ ]:
assert train_df["Survived"].drop_nulls().is_in([0, 1]).all()
assert train_df["Pclass"].drop_nulls().is_in([1, 2, 3]).all()
train_df["Sex"].unique()
train_df["Embarked"].unique()

## Data profiling

In [ ]:
train_df.shape
train_df.null_count()
train_df.describe()
train_df.n_unique()
train_df.head()

## Ingestion report

# Test Dataset

In [ ]:
train.describe()